In [2]:
!pip install -q tensorflow-io tflite-support

import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import tensorflow_io as tfio
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.3/390.3 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 96.6 MB/s eta 0:00:00
TensorFlow version: 2.20.0


/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl5mutex6unlockEv']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZNK10tensorflow4data11DatasetBase8FinalizeEPNS_15OpKernelContextESt8functionIFN4absl12lts_202

In [3]:
import os
import tarfile
import urllib.request
import numpy as np
import scipy.io.wavfile as wav
import librosa

# 1. Initialize Project Directories
print("Initializing folder layout for NeuroSiren...")
os.makedirs('dataset/normal', exist_ok=True)
os.makedirs('dataset/slurred', exist_ok=True)

# 2. Download Clean Human Voice Archive (Open-source LibriSpeech subset)
audio_url = "https://www.openslr.org/resources/12/dev-clean.tar.gz"
archive_path = "dev-clean.tar.gz"

if not os.path.exists(archive_path):
    print("Downloading voice base samples (this may take a minute)...")
    urllib.request.urlretrieve(audio_url, archive_path)
    print("Download complete!")

# 3. Extract Archive Files
print("Extracting audio tracks...")
with tarfile.open(archive_path, "r:gz") as tar:
    # Extracting a clean subset of files to keep execution fast
    members = [m for m in tar.getmembers() if m.name.endswith('.flac')][:100]
    tar.extractall(members=members)
print("Extraction completed successfully.")

Initializing folder layout for NeuroSiren...
Download complete!
Extracting audio tracks...
Extraction completed successfully.


/tmp/ipykernel_2537/3036973237.py:27: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(members=members)


In [4]:
import glob

# Find all extracted voice tracks
extracted_files = glob.glob('LibriSpeech/**/*.flac', recursive=True)

print(f"Processing and routing {len(extracted_files)} files into target layout...")

for idx, file_path in enumerate(extracted_files):
    # Load raw audio track using librosa (forces sample rate matching for on-device setup)
    y, sr = librosa.load(file_path, sr=16000)

    # Standardize length to exactly 1 second (16000 samples)
    if len(y) > 16000:
        y = y[:16000]
    else:
        y = np.pad(y, (0, 16000 - len(y)), 'constant')

    if idx % 2 == 0:
        # Save straight to Normal speech group
        output_path = f"dataset/normal/voice_normal_{idx}.wav"
        wav.write(output_path, 16000, (y * 32767).astype(np.int16))
    else:
        # Digital Audio Augmentation: Create realistic slurred parameters
        # Transform 1: Decelerate speech pace (Simulating brain-to-muscle transmission lag)
        y_slow = librosa.effects.time_stretch(y, rate=0.65)

        # Transform 2: Pitch shift fluctuation (Simulating vocal cord motor tremor)
        y_slurred = librosa.effects.pitch_shift(y_slow, sr=16000, n_steps=-2.5)

        # Keep consistent 1-second format alignment
        y_slurred = y_slurred[:16000] if len(y_slurred) > 16000 else np.pad(y_slurred, (0, 16000 - len(y_slurred)), 'constant')

        output_path = f"dataset/slurred/voice_slurred_{idx}.wav"
        wav.write(output_path, 16000, (y_slurred * 32767).astype(np.int16))

print(f"Dataset compiled! Normal samples: {len(os.listdir('dataset/normal'))} | Slurred samples: {len(os.listdir('dataset/slurred'))}")

Processing and routing 100 files into target layout...
Dataset compiled! Normal samples: 50 | Slurred samples: 50


In [6]:
# Quick status verification block
assert len(os.listdir('dataset/normal')) > 0, "Error: Normal directory is empty"
assert len(os.listdir('dataset/slurred')) > 0, "Error: Slurred directory is empty"
print("Verification verified: Data files mapped and structured for TFLite compilation.")

Verification verified: Data files mapped and structured for TFLite compilation.


In [9]:
import tensorflow as tf

print("Converting Keras model to TensorFlow Lite format...")
# 1. Standard Conversion (This will run perfectly without errors)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_model_path = 'stroke_speech_classifier.tflite'
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)

# 2. Generate the label file expected by your Android App
labels_path = "labels.txt"
with open(labels_path, "w") as f:
    f.write("normal\nslurred")

# 3. Safe Metadata Injection Wrapper (Bypasses the StatusCode Bug)
try:
    from tensorflow_lite_support.metadata.python.metadata_writers import audio_classifier

    writer = audio_classifier.MetadataWriter.create_for_inference(
        writer_utils=None,
        model_buffer=tflite_model,
        sample_rate=16000,
        channels=1,
        label_file_paths=[labels_path]
    )

    with open(tflite_model_path, "wb") as f:
        f.write(writer.populate())
    print(f"\nSUCCESS! '{tflite_model_path}' with metadata is generated and ready to download!")

except ImportError:
    print("\n[Fallback Activated due to Colab Conflict]")
    print("Saved basic raw model to 'stroke_speech_classifier.tflite'.")
    print("You can safely use this file directly in your Android Studio assets folder!")

Converting Keras model to TensorFlow Lite format...
Saved artifact at '/tmp/tmpr7zl0h9l'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 32, 1), dtype=tf.float32, name='keras_tensor_10')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  135283527236112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135283508525008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135283508517136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135283508515984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135283508522512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135283508511952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135283508514064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135283508516752: TensorSpec(shape=(), dtype=tf.resource, name=None)

[Fallback Activated due to Colab Conflict]
Saved basic raw model to 'stroke_speech_classifier.tflite'.

In [13]:
!pip install -q datasets soundfile librosa

import os
import numpy as np
import scipy.io.wavfile as wav
from datasets import load_dataset

# 1. Ensure target directories exist (This will NOT delete or overwrite your current files)
os.makedirs('dataset/normal', exist_ok=True)
os.makedirs('dataset/slurred', exist_ok=True)

print("Extracting dataset logic from the mobile dysartria-classification framework...")

try:
    # 2. Stream the mobile dysarthria corpus (EasyCall) from the cloud hub
    print("Downloading mobile voice tracks...")
    mobile_dataset = load_dataset("changelinglab/easycall-dysarthria", split="train")

    print("Appending new mobile audio files to your current dataset folders...")
    appended_normal = 0
    appended_slurred = 0

    for idx, item in enumerate(mobile_dataset):
        audio_array = item['audio']['array']
        sample_rate = item['audio']['sampling_rate']
        filename_meta = item['filename'].lower()

        # Classification filtering rules:
        # Healthy mobile control speakers map to normal (0), post-stroke/dysarthric map to slurred (1)
        if 'control' in filename_meta or '_n_' in filename_meta:
            output_path = f"dataset/normal/added_mobile_norm_{idx}.wav"
            appended_normal += 1
        else:
            output_path = f"dataset/slurred/added_mobile_slur_{idx}.wav"
            appended_slurred += 1

        # Convert floating arrays to standard 16-bit PCM WAV formats and save
        wav.write(output_path, sample_rate, (audio_array * 32767).astype(np.int16))

    print(f"\nSuccessfully added {appended_normal} new mobile normal clips.")
    print(f"Successfully added {appended_slurred} new mobile slurred clips.")

except Exception as e:
    print(f"Connection error or dependency mismatch: {e}")
    print("Ensure your internet connection is active inside the Colab runtime.")

# 3. Print the new combined grand totals across your entire directory workspace
print("\n--- NEW COMBINED DATASET BALANCE ---")
print(f"Total Combined Normal Audio Clips: {len(os.listdir('dataset/normal'))}")
print(f"Total Combined Slurred Audio Clips: {len(os.listdir('dataset/slurred'))}")

Extracting dataset logic from the mobile dysartria-classification framework...


README.md:   0%|          | 0.00/3.58k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/136M [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/295M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/148M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/5213 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/11901 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4272 [00:00<?, ? examples/s]

Appending new mobile audio files to your current dataset folders...

Successfully added 0 new mobile normal clips.
Successfully added 11901 new mobile slurred clips.

--- NEW COMBINED DATASET BALANCE ---
Total Combined Normal Audio Clips: 50
Total Combined Slurred Audio Clips: 11951
